# Diagnóstico 07 — otimização de memória do Dashboard ENEM

## Objetivo

Validar, em processos independentes, as próximas hipóteses de redução de memória identificadas nos diagnósticos anteriores:

1. leitura PyArrow com `use_threads=False`;
2. comparação dos Memory Pools (`mimalloc`, `system` e `jemalloc`, quando disponíveis);
3. leitura de colunas textuais com `read_dictionary`;
4. redução de tipos numéricos e revisão das colunas;
5. reorganização física dos Parquets, sem sobrescrever os arquivos originais;
6. medição do consumo da aplicação com Gunicorn e um worker.

**Este notebook não altera o loader de produção nem sobrescreve os Parquets originais.** As variantes físicas serão gravadas em uma pasta separada dentro do projeto.

### Metodologia

- Cada medição de leitura é executada em um subprocesso novo, para evitar que a memória retida por uma execução contamine a seguinte.
- As comparações usam as projeções de colunas e os filtros territoriais do loader atual.
- São registradas USS, RSS, tempo, memória do Memory Pool e tamanho profundo dos DataFrames.
- As assinaturas dos DataFrames são calculadas depois da medição principal de carregamento, para reduzir a interferência da validação nos números de pico.
- A mediana das repetições será usada como referência; uma execução isolada não será tratada como resultado conclusivo.

> Execute as seções na ordem. Comece com poucas repetições (`REPETICOES = 2`) e aumente para 3 ou mais se os resultados forem estáveis.

## 0. Configuração do ambiente e localização dos dados

In [1]:
from pathlib import Path
import os
import sys
import json
import time
import gc
import subprocess
import textwrap
import statistics
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import psutil
from IPython.display import display

# O notebook deve ser executado dentro de app_render ou em um subdiretório dele.
CANDIDATOS_ROOT = [Path.cwd(), *Path.cwd().parents]
CANDIDATOS_ROOT.append(Path.home() / "PycharmProjects" / "Projeto_ENEM" / "app_render")

PROJECT_ROOT = next(
    (p.resolve() for p in CANDIDATOS_ROOT if (p / "dash_utils" / "config.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Não encontrei app_render/dash_utils/config.py. "
        "Ajuste PROJECT_ROOT manualmente para a pasta app_render."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import dash_utils.config as config

UF = "PA"
REFERENCIAS = ["ESC", "PROVA"]
REPETICOES = 2
USE_THREADS_CONTROLE = True  # altere após avaliar a etapa 1, se houver ganho consistente

CONSUMO_UF_DIR = Path(config.CONSUMO_DIR) / f"uf={UF}"
DATA_DIR = Path(config.DATA_DIR)
SHAPE_PATH = DATA_DIR / "shapefiles" / f"{UF}_Municipios_2025" / f"{UF}_Municipios_2025.shp"

ARQUIVOS_ORIGINAIS = {
    "painel": CONSUMO_UF_DIR / "painel_municipal.parquet",
    "faixa": CONSUMO_UF_DIR / "faixa_etaria_municipal.parquet",
    "grupo": CONSUMO_UF_DIR / "desempenho_grupos_municipal.parquet",
    "dsp": CONSUMO_UF_DIR / "desempenho_sexo_faixa_etaria_municipal.parquet",
}

EXPERIMENT_DIR = PROJECT_ROOT / "_diagnosticos_memoria_07"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

COLUNAS = {
    "painel": [
        "ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL",
        "n_participantes", "n_masculino", "n_feminino",
        "n_faixa_etaria_informada", "n_escola_federal",
        "n_escola_estadual", "n_escola_municipal", "n_escola_privada",
        "n_escola_nao_informada", "media_redacao", "n_validos_redacao",
        "media_objetiva", "n_validos_objetiva", "media_cn", "n_validos_cn",
        "media_ch", "n_validos_ch", "media_lc", "n_validos_lc",
        "media_mt", "n_validos_mt",
    ],
    "faixa": [
        "ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL",
        "TP_FAIXA_ETARIA", "n_participantes",
    ],
    "grupo": [
        "ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL",
        "tipo_grupo", "grupo", "disciplina", "media", "n_validos",
        "mediana", "std", "q1", "q3",
    ],
    "dsp": [
        "ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL",
        "sexo", "faixa_etaria", "disciplina", "n_participantes",
        "n_validos", "media", "std",
    ],
}

faltantes = [str(p) for p in [*ARQUIVOS_ORIGINAIS.values(), SHAPE_PATH] if not p.exists()]
if faltantes:
    raise FileNotFoundError("Arquivos ausentes:\n" + "\n".join(faltantes))

print("Raiz:", PROJECT_ROOT)
print("UF:", UF)
print("PyArrow:", pa.__version__)
print("Memory Pool do kernel:", pa.default_memory_pool().backend_name)
print("Diretório de resultados:", EXPERIMENT_DIR)

Raiz: /home/akel/PycharmProjects/Projeto_ENEM/app_render
UF: PA
PyArrow: 26.0.1
Memory Pool do kernel: mimalloc
Diretório de resultados: /home/akel/PycharmProjects/Projeto_ENEM/app_render/_diagnosticos_memoria_07


### Inventário dos arquivos Parquet

In [2]:
linhas = []
for nome, caminho in ARQUIVOS_ORIGINAIS.items():
    pf = pq.ParquetFile(caminho)
    schema = pf.schema_arrow
    linhas.append({
        "base": nome,
        "arquivo_MB": round(caminho.stat().st_size / 1024**2, 2),
        "linhas": pf.metadata.num_rows,
        "row_groups": pf.metadata.num_row_groups,
        "colunas_fisicas": len(schema.names),
        "colunas_projetadas": len(COLUNAS[nome]),
        "colunas_fora_projecao": ", ".join(c for c in schema.names if c not in COLUNAS[nome]),
    })

inventario = pd.DataFrame(linhas)
display(inventario)

,base,arquivo_MB,linhas,row_groups,colunas_fisicas,colunas_projetadas,colunas_fora_projecao
0,painel,1.19,6464,1,82,24,"NO_MUNICIPIO_TERRITORIO, pct_masculino, pct_fe..."
1,faixa,0.14,63836,1,7,5,"NO_MUNICIPIO_TERRITORIO, base_origem"
2,grupo,8.57,337214,1,16,12,"NO_MUNICIPIO_TERRITORIO, n_zeros, media_positi..."
3,dsp,10.10,451205,1,17,10,"NO_MUNICIPIO_TERRITORIO, n_zeros, media_positi..."


## 1. Worker de leitura isolada

O worker abaixo replica a projeção, o filtro territorial, a leitura do shapefile e a preparação municipal do loader atual. Ele recebe os parâmetros por JSON e devolve apenas métricas e assinaturas.

**Importante:** a variável `ARROW_DEFAULT_MEMORY_POOL` será definida no ambiente do subprocesso antes da importação do PyArrow. Isso é necessário para que a comparação dos alocadores seja válida.

In [3]:
WORKER_SCRIPT_SOURCE = r"""
import os, sys, json, gc, time
from pathlib import Path

payload = json.load(sys.stdin)
project_root = Path(payload["project_root"])
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import geopandas as gpd
import psutil

import dash_utils.config as config

UF = payload["uf"]
ref = payload["referencia"]
use_threads = bool(payload.get("use_threads", True))
read_dictionary = bool(payload.get("read_dictionary", False))
mode = payload.get("mode", "load")
layout = payload.get("layout", "original")
row_group_size = int(payload.get("row_group_size", 100000))
experiment_dir = Path(payload["experiment_dir"])

cols = {
    "painel": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL", "n_participantes", "n_masculino", "n_feminino", "n_faixa_etaria_informada", "n_escola_federal", "n_escola_estadual", "n_escola_municipal", "n_escola_privada", "n_escola_nao_informada", "media_redacao", "n_validos_redacao", "media_objetiva", "n_validos_objetiva", "media_cn", "n_validos_cn", "media_ch", "n_validos_ch", "media_lc", "n_validos_lc", "media_mt", "n_validos_mt"],
    "faixa": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL", "TP_FAIXA_ETARIA", "n_participantes"],
    "grupo": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL", "tipo_grupo", "grupo", "disciplina", "media", "n_validos", "mediana", "std", "q1", "q3"],
    "dsp": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL", "sexo", "faixa_etaria", "disciplina", "n_participantes", "n_validos", "media", "std"],
}

originais = {
    "painel": Path(config.CONSUMO_DIR) / f"uf={UF}" / "painel_municipal.parquet",
    "faixa": Path(config.CONSUMO_DIR) / f"uf={UF}" / "faixa_etaria_municipal.parquet",
    "grupo": Path(config.CONSUMO_DIR) / f"uf={UF}" / "desempenho_grupos_municipal.parquet",
    "dsp": Path(config.CONSUMO_DIR) / f"uf={UF}" / "desempenho_sexo_faixa_etaria_municipal.parquet",
}

shape_path = Path(config.DATA_DIR) / "shapefiles" / f"{UF}_Municipios_2025" / f"{UF}_Municipios_2025.shp"

processo = psutil.Process()
def mem():
    m = processo.memory_full_info()
    return {"rss_mb": m.rss / 1024**2, "uss_mb": getattr(m, "uss", float("nan")) / 1024**2}

def snapshot(etapa):
    return {"etapa": etapa, **mem(), "arrow_pool_mb": pa.default_memory_pool().bytes_allocated() / 1024**2}

def normalizar_nome(nome):
    import unicodedata
    if not isinstance(nome, str):
        return nome
    return unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode("ascii").upper().strip()

def assinatura(df):
    h = pd.util.hash_pandas_object(df, index=False).to_numpy(dtype=np.uint64)
    return {
        "linhas": int(len(df)),
        "hash_soma_uint64": int(h.sum(dtype=np.uint64)),
        "hash_xor_uint64": int(np.bitwise_xor.reduce(h, initial=np.uint64(0))),
    }

def caminho_leitura(base):
    if layout == "original":
        return originais[base], True
    base_dir = experiment_dir / "parquet_variants" / base
    if layout.startswith("ordenado_rg_"):
        return base_dir / f"{layout}.parquet", True
    if layout == "separado_rg_100000":
        return base_dir / "separado_rg_100000" / f"{ref}.parquet", False
    raise ValueError(f"Layout desconhecido: {layout}")

pool = pa.default_memory_pool()
backend = pool.backend_name
inicio_total = time.perf_counter()
medidas = [snapshot("baseline_apos_imports")]
frames = {}

for base in ["painel", "faixa", "grupo", "dsp"]:
    caminho, aplicar_filtro = caminho_leitura(base)
    filtros = None
    if aplicar_filtro:
        valores = ["ESC", "RESIDENCIA"] if ref == "ESC" else ["PROVA"]
        filtros = [("TIPO_REFERENCIA_TERRITORIAL", "in", valores)]
    kwargs = {"columns": cols[base], "use_threads": use_threads, "filters": filtros}
    if read_dictionary:
        dict_cols = payload.get("dictionary_columns", {}).get(base, [])
        if dict_cols:
            kwargs["read_dictionary"] = dict_cols
    t0 = time.perf_counter()
    frames[base] = pd.read_parquet(caminho, engine="pyarrow", **kwargs)
    medidas.append(snapshot(f"apos_leitura_{base}"))

shape = gpd.read_file(shape_path)
mapa = {
    str(cd): normalizar_nome(nm)
    for cd, nm in zip(shape["CD_MUN"], shape["NM_MUN"])
}

for base, df in list(frames.items()):
    df["TIPO_REFERENCIA_TERRITORIAL"] = df["TIPO_REFERENCIA_TERRITORIAL"].replace("RESIDENCIA", "ESC")
    df = df.loc[df["TIPO_REFERENCIA_TERRITORIAL"].eq(ref)]
    df["NO_MUNICIPIO_TERRITORIO"] = df["CO_MUNICIPIO_TERRITORIO"].astype(str).map(mapa)
    df = df.dropna(subset=["NO_MUNICIPIO_TERRITORIO"]).reset_index(drop=True)
    frames[base] = df

medidas.append(snapshot("apos_shapefile_e_preparacao"))

# A medição abaixo é anterior à geração das assinaturas.
metricas_apos_carga = snapshot("apos_carga_pre_validacao")
deep = {nome: float(df.memory_usage(deep=True).sum() / 1024**2) for nome, df in frames.items()}

if mode == "downcast":
    relatorio_downcast = {}
    for nome, df in frames.items():
        antes = float(df.memory_usage(deep=True).sum() / 1024**2)
        alteracoes = []
        erros_float = []
        for coluna in df.columns:
            serie = df[coluna]
            dtype_original = str(serie.dtype)
            if pd.api.types.is_integer_dtype(serie.dtype) and dtype_original in ("int64", "uint64"):
                nova = pd.to_numeric(serie, downcast="integer")
                if str(nova.dtype) != dtype_original:
                    df[coluna] = nova
                    alteracoes.append({"coluna": coluna, "de": dtype_original, "para": str(nova.dtype)})
            elif pd.api.types.is_float_dtype(serie.dtype) and dtype_original == "float64":
                original = serie.to_numpy(dtype="float64", copy=True)
                nova = serie.astype("float32")
                if len(original):
                    erro = float(np.nanmax(np.abs(original - nova.to_numpy(dtype="float64"))))
                else:
                    erro = 0.0
                erros_float.append({"coluna": coluna, "erro_abs_max": erro})
                df[coluna] = nova
                alteracoes.append({"coluna": coluna, "de": dtype_original, "para": "float32"})
        depois = float(df.memory_usage(deep=True).sum() / 1024**2)
        schema_fonte = pq.read_schema(originais[nome]).names
        relatorio_downcast[nome] = {
            "deep_antes_mb": antes,
            "deep_depois_mb": depois,
            "reducao_pct": (1 - depois / antes) * 100 if antes else 0,
            "alteracoes_dtype": alteracoes,
            "erros_float32": erros_float,
            "colunas_fonte_nao_projetadas": [c for c in schema_fonte if c not in cols[nome]],
        }
    print(json.dumps({"status": "ok", "backend": backend, "downcast": relatorio_downcast}, ensure_ascii=False))
    raise SystemExit(0)

# Assinaturas são calculadas depois da medição principal.
assinaturas = {nome: assinatura(df) for nome, df in frames.items()}

for nome in list(frames):
    del frames[nome]
del frames, shape, mapa
gc.collect()
medidas.append(snapshot("apos_del_gc_apos_validacao"))

pool_info = {
    "backend": backend,
    "bytes_allocated_mb": pool.bytes_allocated() / 1024**2,
    "max_memory_mb": pool.max_memory() / 1024**2 if hasattr(pool, "max_memory") else None,
}
resultado = {
    "status": "ok",
    "uf": UF,
    "referencia": ref,
    "layout": layout,
    "use_threads": use_threads,
    "read_dictionary": read_dictionary,
    "memory_pool_env": os.environ.get("ARROW_DEFAULT_MEMORY_POOL", "(não definido)"),
    "pool": pool_info,
    "tempo_total_s": time.perf_counter() - inicio_total,
    "medidas": medidas,
    "apos_carga": metricas_apos_carga,
    "deep_mb": deep,
    "assinaturas": assinaturas,
}
print(json.dumps(resultado, ensure_ascii=False))
"""

### Funções para executar e resumir os subprocessos

In [4]:
WORKER_SCRIPT = textwrap.dedent(WORKER_SCRIPT_SOURCE)

def executar_worker(
    referencia,
    use_threads=True,
    read_dictionary=False,
    dictionary_columns=None,
    memory_pool=None,
    mode="load",
    layout="original",
    row_group_size=100000,
    timeout=900,
):
    payload = {
        "project_root": str(PROJECT_ROOT),
        "experiment_dir": str(EXPERIMENT_DIR),
        "uf": UF,
        "referencia": referencia,
        "use_threads": use_threads,
        "read_dictionary": read_dictionary,
        "dictionary_columns": dictionary_columns or {},
        "mode": mode,
        "layout": layout,
        "row_group_size": row_group_size,
    }
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    if memory_pool:
        env["ARROW_DEFAULT_MEMORY_POOL"] = memory_pool
    else:
        env.pop("ARROW_DEFAULT_MEMORY_POOL", None)

    proc = subprocess.run(
        [sys.executable, "-c", WORKER_SCRIPT],
        input=json.dumps(payload),
        text=True,
        capture_output=True,
        cwd=PROJECT_ROOT,
        env=env,
        timeout=timeout,
    )
    if proc.returncode != 0:
        raise RuntimeError(
            f"Worker falhou ({proc.returncode}) para {referencia}/{layout}.\n"
            f"STDOUT:\n{proc.stdout[-4000:]}\nSTDERR:\n{proc.stderr[-6000:]}"
        )
    linhas_json = [linha for linha in proc.stdout.splitlines() if linha.strip().startswith("{")]
    if not linhas_json:
        raise RuntimeError(f"Worker não retornou JSON. Saída:\n{proc.stdout}\n{proc.stderr}")
    return json.loads(linhas_json[-1])

def executar_repeticoes(cenario, referencias=REFERENCIAS, repeticoes=REPETICOES, **kwargs):
    resultados = []
    for ref in referencias:
        for repeticao in range(1, repeticoes + 1):
            print(f"{cenario}: {ref} — repetição {repeticao}/{repeticoes}")
            resultado = executar_worker(referencia=ref, **kwargs)
            resultado["cenario"] = cenario
            resultado["repeticao"] = repeticao
            resultados.append(resultado)
    return resultados

def resumir_resultados(resultados):
    linhas = []
    for r in resultados:
        m = r["apos_carga"]
        fim = r["medidas"][-1]
        linhas.append({
            "cenario": r.get("cenario", r.get("layout")),
            "referencia": r["referencia"],
            "repeticao": r.get("repeticao"),
            "USS_apos_carga_MB": m["uss_mb"],
            "RSS_apos_carga_MB": m["rss_mb"],
            "USS_apos_gc_MB": fim["uss_mb"],
            "Arrow_pool_MB": r["pool"]["bytes_allocated_mb"],
            "tempo_s": r["tempo_total_s"],
            "deep_total_MB": sum(r["deep_mb"].values()),
            "backend": r["pool"]["backend"],
        })
    df = pd.DataFrame(linhas)
    if df.empty:
        return df
    resumo = (
        df.groupby(["cenario", "referencia"], as_index=False)
        .agg(
            USS_mediana_MB=("USS_apos_carga_MB", "median"),
            USS_min_MB=("USS_apos_carga_MB", "min"),
            USS_max_MB=("USS_apos_carga_MB", "max"),
            RSS_mediana_MB=("RSS_apos_carga_MB", "median"),
            USS_apos_gc_mediana_MB=("USS_apos_gc_MB", "median"),
            Arrow_pool_mediana_MB=("Arrow_pool_MB", "median"),
            tempo_mediano_s=("tempo_s", "median"),
            deep_total_mediana_MB=("deep_total_MB", "median"),
            n=("repeticao", "count"),
        )
    )
    return df, resumo

## 2. Experimento 1 — `use_threads=True` × `use_threads=False`

A comparação mantém constantes os arquivos, as projeções, os filtros e a preparação territorial. Cada execução parte de um processo novo.

In [5]:
resultados_threads = []
for ref in REFERENCIAS:
    for cenario, flag_threads in [("threads_true", True), ("threads_false", False)]:
        for repeticao in range(1, REPETICOES + 1):
            print(f"{ref} | {cenario} | repetição {repeticao}/{REPETICOES}")
            r = executar_worker(referencia=ref, use_threads=flag_threads)
            r["cenario"] = cenario
            r["repeticao"] = repeticao
            resultados_threads.append(r)

threads_detalhe, threads_resumo = resumir_resultados(resultados_threads)
display(threads_detalhe)
display(threads_resumo)

ESC | threads_true | repetição 1/2
ESC | threads_true | repetição 2/2
ESC | threads_false | repetição 1/2
ESC | threads_false | repetição 2/2
PROVA | threads_true | repetição 1/2
PROVA | threads_true | repetição 2/2
PROVA | threads_false | repetição 1/2
PROVA | threads_false | repetição 2/2


,cenario,referencia,repeticao,USS_apos_carga_MB,RSS_apos_carga_MB,USS_apos_gc_MB,Arrow_pool_MB,tempo_s,deep_total_MB,backend
0,threads_true,ESC,1,331.722656,430.398438,324.804688,19.972229,0.680629,58.853294,mimalloc
1,threads_true,ESC,2,365.417969,464.011719,358.199219,19.972229,0.448939,58.853294,mimalloc
2,threads_false,ESC,1,326.054688,424.664062,318.871094,19.972229,0.457550,58.853294,mimalloc
3,threads_false,ESC,2,314.027344,412.710938,307.089844,19.972229,0.440829,58.853294,mimalloc
4,threads_true,PROVA,1,329.820312,427.984375,328.875000,20.809082,0.357011,42.586669,mimalloc
5,threads_true,PROVA,2,319.550781,417.726562,318.609375,20.809082,0.395891,42.586669,mimalloc
6,threads_false,PROVA,1,270.648438,368.839844,271.683594,20.809082,0.416592,42.586669,mimalloc
7,threads_false,PROVA,2,288.687500,386.703125,289.699219,20.809082,0.406855,42.586669,mimalloc


,cenario,referencia,USS_mediana_MB,USS_min_MB,USS_max_MB,RSS_mediana_MB,USS_apos_gc_mediana_MB,Arrow_pool_mediana_MB,tempo_mediano_s,deep_total_mediana_MB,n
0,threads_false,ESC,320.041016,314.027344,326.054688,418.687500,312.980469,19.972229,0.449190,58.853294,2
1,threads_false,PROVA,279.667969,270.648438,288.687500,377.771484,280.691406,20.809082,0.411724,42.586669,2
2,threads_true,ESC,348.570312,331.722656,365.417969,447.205078,341.501953,19.972229,0.564784,58.853294,2
3,threads_true,PROVA,324.685547,319.550781,329.820312,422.855469,323.742188,20.809082,0.376451,42.586669,2


### Interpretação do experimento 1

Considere a mediana de USS após o carregamento, e não apenas uma execução. Verifique também RSS e tempo. Só altere `USE_THREADS_CONTROLE` se a redução de memória aparecer nas duas referências, ou se houver uma justificativa operacional clara para otimizar apenas uma delas.

In [6]:
# Defina manualmente após avaliar a tabela acima.
# Exemplo: USE_THREADS_CONTROLE = False
print("USE_THREADS_CONTROLE atual:", USE_THREADS_CONTROLE)

USE_THREADS_CONTROLE atual: True


## 3. Experimento 2 — Memory Pool

O backend é configurado no ambiente do subprocesso antes da importação do PyArrow. Alguns builds podem não oferecer todos os backends; nesse caso, a coluna `backend` mostrará o que foi efetivamente selecionado.

In [7]:
ok..resultados_pools = []

for pool_solicitado in CANDIDATOS_POOLS:
    for ref in REFERENCIAS:
        for repeticao in range(1, REPETICOES + 1):
            print(f"Pool={pool_solicitado} | {ref} | repetição {repeticao}/{REPETICOES}")
            r = executar_worker(
                referencia=ref,
                use_threads=USE_THREADS_CONTROLE,
                memory_pool=pool_solicitado,
            )
            r["cenario"] = f"pool_{pool_solicitado}"
            r["pool_solicitado"] = pool_solicitado
            r["repeticao"] = repeticao
            r["pool_disponivel"] = r["pool"]["backend"] == pool_solicitado
            resultados_pools.append(r)

pools_detalhe, pools_resumo = resumir_resultados(resultados_pools)
display(pd.DataFrame([
    {"solicitado": r["pool_solicitado"], "backend_real": r["pool"]["backend"],
     "referencia": r["referencia"], "repeticao": r["repeticao"],
     "USS_MB": r["apos_carga"]["uss_mb"], "USS_apos_gc_MB": r["medidas"][-1]["uss_mb"],
     "pool_MB": r["pool"]["bytes_allocated_mb"], "tempo_s": r["tempo_total_s"],
     "disponivel": r["pool_disponivel"]}
    for r in resultados_pools
]))
display(pools_resumo)

Pool=mimalloc | ESC | repetição 1/2
Pool=mimalloc | ESC | repetição 2/2
Pool=mimalloc | PROVA | repetição 1/2
Pool=mimalloc | PROVA | repetição 2/2
Pool=system | ESC | repetição 1/2
Pool=system | ESC | repetição 2/2
Pool=system | PROVA | repetição 1/2
Pool=system | PROVA | repetição 2/2
Pool=jemalloc | ESC | repetição 1/2
Pool=jemalloc | ESC | repetição 2/2
Pool=jemalloc | PROVA | repetição 1/2
Pool=jemalloc | PROVA | repetição 2/2


,solicitado,backend_real,referencia,repeticao,USS_MB,USS_apos_gc_MB,pool_MB,tempo_s,disponivel
0,mimalloc,mimalloc,ESC,1,356.484375,349.289062,19.972229,0.606532,True
1,mimalloc,mimalloc,ESC,2,357.464844,350.289062,19.972229,0.575687,True
2,mimalloc,mimalloc,PROVA,1,332.964844,331.933594,20.809082,0.360500,True
3,mimalloc,mimalloc,PROVA,2,328.914062,327.968750,20.809082,0.331134,True
4,system,system,ESC,1,216.835938,211.656250,19.972229,0.525770,True
5,system,system,ESC,2,245.707031,238.757812,19.972229,0.516008,True
6,system,system,PROVA,1,174.210938,175.050781,20.809082,0.316845,True
7,system,system,PROVA,2,182.121094,182.996094,20.809082,0.315823,True
8,jemalloc,jemalloc,ESC,1,314.746094,307.582031,19.972229,0.450310,True
9,jemalloc,jemalloc,ESC,2,315.558594,308.660156,19.972229,0.472559,True


,cenario,referencia,USS_mediana_MB,USS_min_MB,USS_max_MB,RSS_mediana_MB,USS_apos_gc_mediana_MB,Arrow_pool_mediana_MB,tempo_mediano_s,deep_total_mediana_MB,n
0,pool_jemalloc,ESC,315.152344,314.746094,315.558594,413.705078,308.121094,19.972229,0.461434,58.853294,2
1,pool_jemalloc,PROVA,226.216797,225.351562,227.082031,324.462891,228.369141,20.809082,0.338769,42.586669,2
2,pool_mimalloc,ESC,356.974609,356.484375,357.464844,455.779297,349.789062,19.972229,0.591109,58.853294,2
3,pool_mimalloc,PROVA,330.939453,328.914062,332.964844,429.050781,329.951172,20.809082,0.345817,42.586669,2
4,pool_system,ESC,231.271484,216.835938,245.707031,329.912109,225.207031,19.972229,0.520889,58.853294,2
5,pool_system,PROVA,178.166016,174.210938,182.121094,276.470703,179.023438,20.809082,0.316334,42.586669,2


**Critério:** só compare um backend quando `backend_real` for igual ao solicitado. A variável de ambiente pode ser ignorada se o backend não estiver disponível ou se o pool já tiver sido inicializado antes da configuração.

## 4. Experimento 3 — `read_dictionary`

O PyArrow permite solicitar `DictionaryArray` diretamente durante a leitura de colunas armazenadas como `BYTE_ARRAY`. Aqui, o notebook identifica as colunas textuais projetadas no schema físico de cada arquivo. A leitura comum e a leitura com dicionário serão comparadas com o mesmo filtro e a mesma projeção.

In [8]:
CANDIDATAS_DICIONARIO = {
    "painel": ["TIPO_REFERENCIA_TERRITORIAL"],
    "faixa": ["TIPO_REFERENCIA_TERRITORIAL"],
    "grupo": ["TIPO_REFERENCIA_TERRITORIAL", "tipo_grupo", "grupo", "disciplina"],
    "dsp": ["TIPO_REFERENCIA_TERRITORIAL", "sexo", "faixa_etaria", "disciplina"],
}

DICTIONARY_COLUMNS = {}
for base, caminho in ARQUIVOS_ORIGINAIS.items():
    schema = pq.read_schema(caminho)
    DICTIONARY_COLUMNS[base] = [
        nome for nome in CANDIDATAS_DICIONARIO[base]
        if nome in schema.names and str(schema.field(nome).type) in ("string", "large_string", "binary", "large_binary")
    ]

print("Colunas solicitadas como dicionário:")
print(json.dumps(DICTIONARY_COLUMNS, indent=2, ensure_ascii=False))

Colunas solicitadas como dicionário:
{
  "painel": [
    "TIPO_REFERENCIA_TERRITORIAL"
  ],
  "faixa": [
    "TIPO_REFERENCIA_TERRITORIAL"
  ],
  "grupo": [
    "TIPO_REFERENCIA_TERRITORIAL",
    "tipo_grupo",
    "grupo",
    "disciplina"
  ],
  "dsp": [
    "TIPO_REFERENCIA_TERRITORIAL",
    "sexo",
    "faixa_etaria",
    "disciplina"
  ]
}


In [9]:
resultados_dictionary = []
for ref in REFERENCIAS:
    for cenario, usar_dictionary in [("sem_dictionary", False), ("com_dictionary", True)]:
        for repeticao in range(1, REPETICOES + 1):
            print(f"{ref} | {cenario} | repetição {repeticao}/{REPETICOES}")
            r = executar_worker(
                referencia=ref,
                use_threads=USE_THREADS_CONTROLE,
                read_dictionary=usar_dictionary,
                dictionary_columns=DICTIONARY_COLUMNS,
            )
            r["cenario"] = cenario
            r["repeticao"] = repeticao
            resultados_dictionary.append(r)

dictionary_detalhe, dictionary_resumo = resumir_resultados(resultados_dictionary)
display(dictionary_detalhe)
display(dictionary_resumo)

ESC | sem_dictionary | repetição 1/2
ESC | sem_dictionary | repetição 2/2
ESC | com_dictionary | repetição 1/2
ESC | com_dictionary | repetição 2/2
PROVA | sem_dictionary | repetição 1/2
PROVA | sem_dictionary | repetição 2/2
PROVA | com_dictionary | repetição 1/2
PROVA | com_dictionary | repetição 2/2


,cenario,referencia,repeticao,USS_apos_carga_MB,RSS_apos_carga_MB,USS_apos_gc_MB,Arrow_pool_MB,tempo_s,deep_total_MB,backend
0,sem_dictionary,ESC,1,371.882812,470.484375,364.921875,19.972229,0.643311,58.853294,mimalloc
1,sem_dictionary,ESC,2,367.582031,466.187500,360.414062,19.972229,0.543268,58.853294,mimalloc
2,com_dictionary,ESC,1,311.953125,410.355469,304.734375,8.246399,0.349786,39.023049,mimalloc
3,com_dictionary,ESC,2,315.937500,414.406250,308.769531,8.246399,0.360957,39.023049,mimalloc
4,sem_dictionary,PROVA,1,330.925781,429.054688,329.976562,20.809082,0.315403,42.586669,mimalloc
5,sem_dictionary,PROVA,2,320.781250,418.968750,319.792969,20.809082,0.319791,42.586669,mimalloc
6,com_dictionary,PROVA,1,288.207031,386.699219,287.226562,11.770630,0.266724,27.659009,mimalloc
7,com_dictionary,PROVA,2,280.121094,378.789062,279.136719,11.770630,0.281062,27.659009,mimalloc


,cenario,referencia,USS_mediana_MB,USS_min_MB,USS_max_MB,RSS_mediana_MB,USS_apos_gc_mediana_MB,Arrow_pool_mediana_MB,tempo_mediano_s,deep_total_mediana_MB,n
0,com_dictionary,ESC,313.945312,311.953125,315.937500,412.380859,306.751953,8.246399,0.355371,39.023049,2
1,com_dictionary,PROVA,284.164062,280.121094,288.207031,382.744141,283.181641,11.770630,0.273893,27.659009,2
2,sem_dictionary,ESC,369.732422,367.582031,371.882812,468.335938,362.667969,19.972229,0.593290,58.853294,2
3,sem_dictionary,PROVA,325.853516,320.781250,330.925781,424.011719,324.884766,20.809082,0.317597,42.586669,2


**Atenção:** menor tamanho profundo do pandas ou menor quantidade de buffers Arrow não significa, por si só, menor USS. A decisão deve considerar USS, RSS, tempo, tipos resultantes e compatibilidade com os filtros e callbacks do dashboard.

## 5. Experimento 4 — downcasting e revisão das colunas

Este teste não tenta reduzir o pico da leitura. Ele estima a economia no tamanho profundo dos DataFrames depois de carregados e mede o erro absoluto máximo de conversão `float64 → float32`.

Os inteiros são reduzidos com `pd.to_numeric(..., downcast="integer")`, quando possível. Os floats `float64` são convertidos para `float32`. Nenhum Parquet de produção é modificado nesta etapa.

In [10]:
resultados_downcast = []
for ref in REFERENCIAS:
    print(f"Downcasting | {ref}")
    r = executar_worker(
        referencia=ref,
        use_threads=USE_THREADS_CONTROLE,
        mode="downcast",
    )
    r["referencia"] = ref
    resultados_downcast.append(r)

linhas_downcast = []
for r in resultados_downcast:
    for base, info in r["downcast"].items():
        linhas_downcast.append({
            "referencia": r["referencia"],
            "base": base,
            "deep_antes_MB": info["deep_antes_mb"],
            "deep_depois_MB": info["deep_depois_mb"],
            "reducao_pct": info["reducao_pct"],
            "n_alteracoes_dtype": len(info["alteracoes_dtype"]),
            "erro_float32_max": max([x["erro_abs_max"] for x in info["erros_float32"]], default=0),
            "colunas_fora_projecao": ", ".join(info["colunas_fonte_nao_projetadas"]),
        })

df_downcast = pd.DataFrame(linhas_downcast)
display(df_downcast)

Downcasting | ESC
Downcasting | PROVA


,referencia,base,deep_antes_MB,deep_depois_MB,reducao_pct,n_alteracoes_dtype,erro_float32_max,colunas_fora_projecao
0,ESC,painel,0.795334,0.580996,26.949475,14,0.000030,"NO_MUNICIPIO_TERRITORIO, pct_masculino, pct_fe..."
1,ESC,faixa,2.638544,2.419189,8.313472,1,0.000000,"NO_MUNICIPIO_TERRITORIO, base_origem"
2,ESC,grupo,25.655953,21.032851,18.019608,6,0.000030,"NO_MUNICIPIO_TERRITORIO, n_zeros, media_positi..."
3,ESC,dsp,29.763463,24.986775,16.048830,4,0.000030,"NO_MUNICIPIO_TERRITORIO, n_zeros, media_positi..."
4,PROVA,painel,0.295180,0.223670,24.225976,14,0.000031,"NO_MUNICIPIO_TERRITORIO, pct_masculino, pct_fe..."
5,PROVA,faixa,1.626272,1.493652,8.154838,1,0.000000,"NO_MUNICIPIO_TERRITORIO, base_origem"
6,PROVA,grupo,18.249453,15.232470,16.531910,6,0.000031,"NO_MUNICIPIO_TERRITORIO, n_zeros, media_positi..."
7,PROVA,dsp,22.415764,18.840553,15.949537,4,0.000031,"NO_MUNICIPIO_TERRITORIO, n_zeros, media_positi..."


### Detalhamento das alterações propostas

Antes de aceitar `float32`, confira a coluna `erro_float32_max` e as tolerâncias exigidas pelos gráficos e indicadores. A lista `colunas_fora_projecao` deve ser revisada no código das páginas: não descarte uma coluna apenas porque ela não aparece no loader atual sem verificar seu uso em callbacks ou funções auxiliares.

In [11]:
for r in resultados_downcast:
    print(f"\nReferência: {r['referencia']}")
    for base, info in r["downcast"].items():
        print(f"\n[{base}]")
        display(pd.DataFrame(info["alteracoes_dtype"]))
        display(pd.DataFrame(info["erros_float32"]))


Referência: ESC

[painel]


,coluna,de,para
0,n_participantes,int64,int16
1,n_faixa_etaria_informada,int64,int16
2,media_redacao,float64,float32
3,n_validos_redacao,int64,int16
4,media_objetiva,float64,float32
5,n_validos_objetiva,float64,float32
6,media_cn,float64,float32
7,n_validos_cn,float64,float32
8,media_ch,float64,float32
9,n_validos_ch,float64,float32


,coluna,erro_abs_max
0,media_redacao,0.000030
1,media_objetiva,0.000002
2,n_validos_objetiva,0.000000
3,media_cn,0.000029
4,n_validos_cn,0.000000
5,media_ch,0.000030
6,n_validos_ch,0.000000
7,media_lc,0.000028
8,n_validos_lc,0.000000
9,media_mt,0.000030



[faixa]


,coluna,de,para
0,n_participantes,int64,int16


""


... (14 outputs left out)

,coluna,erro_abs_max
0,media,0.000031
1,mediana,0.000024
2,std,0.000030
3,q1,0.000024
4,q3,0.000024



[dsp]


,coluna,de,para
0,n_participantes,int64,int16
1,n_validos,int64,int16
2,media,float64,float32
3,std,float64,float32


,coluna,erro_abs_max
0,media,0.000031
1,std,0.000030


## 6. Experimento 5 — reorganização física dos Parquets

Serão geradas variantes em `_diagnosticos_memoria_07/parquet_variants`, sem substituir os arquivos atuais:

- `ordenado_rg_50000`: arquivo ordenado por referência, Row Groups de 50 mil linhas;
- `ordenado_rg_100000`: ordenado, Row Groups de 100 mil linhas;
- `ordenado_rg_250000`: ordenado, Row Groups de 250 mil linhas;
- `separado_rg_100000`: arquivos independentes para ESC e PROVA, com `RESIDENCIA` normalizada para ESC.

A escrita usa as colunas projetadas pelo dashboard e mantém os tipos originais. O objetivo é isolar o efeito da organização física, sem misturar inicialmente com downcasting.

In [12]:
WRITER_SCRIPT_SOURCE = r"""
import sys, json
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import dash_utils.config as config

payload = json.load(sys.stdin)
root = Path(payload["project_root"])
experiment_dir = Path(payload["experiment_dir"])
uf = payload["uf"]
threads = bool(payload["use_threads"])

cols = {
    "painel": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL", "n_participantes", "n_masculino", "n_feminino", "n_faixa_etaria_informada", "n_escola_federal", "n_escola_estadual", "n_escola_municipal", "n_escola_privada", "n_escola_nao_informada", "media_redacao", "n_validos_redacao", "media_objetiva", "n_validos_objetiva", "media_cn", "n_validos_cn", "media_ch", "n_validos_ch", "media_lc", "n_validos_lc", "media_mt", "n_validos_mt"],
    "faixa": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL", "TP_FAIXA_ETARIA", "n_participantes"],
    "grupo": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL", "tipo_grupo", "grupo", "disciplina", "media", "n_validos", "mediana", "std", "q1", "q3"],
    "dsp": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL", "sexo", "faixa_etaria", "disciplina", "n_participantes", "n_validos", "media", "std"],
}
originais = {
    "painel": Path(config.CONSUMO_DIR) / f"uf={uf}" / "painel_municipal.parquet",
    "faixa": Path(config.CONSUMO_DIR) / f"uf={uf}" / "faixa_etaria_municipal.parquet",
    "grupo": Path(config.CONSUMO_DIR) / f"uf={uf}" / "desempenho_grupos_municipal.parquet",
    "dsp": Path(config.CONSUMO_DIR) / f"uf={uf}" / "desempenho_sexo_faixa_etaria_municipal.parquet",
}

row_groups = [50000, 100000, 250000]
resumo = []
for base, origem in originais.items():
    print(f"Preparando {base} ...", flush=True)
    df = pd.read_parquet(origem, engine="pyarrow", columns=cols[base], use_threads=threads)
    df["TIPO_REFERENCIA_TERRITORIAL"] = df["TIPO_REFERENCIA_TERRITORIAL"].replace("RESIDENCIA", "ESC")
    df = df.loc[df["TIPO_REFERENCIA_TERRITORIAL"].isin(["ESC", "PROVA"])].copy()

    for rg in row_groups:
        saida_dir = experiment_dir / "parquet_variants" / base
        saida_dir.mkdir(parents=True, exist_ok=True)
        destino = saida_dir / f"ordenado_rg_{rg}.parquet"
        ordenado = df.sort_values("TIPO_REFERENCIA_TERRITORIAL", kind="mergesort")
        tabela = pa.Table.from_pandas(ordenado, preserve_index=False)
        pq.write_table(
            tabela, destino, row_group_size=rg,
            compression="snappy", use_dictionary=True,
        )
        meta = pq.ParquetFile(destino).metadata
        resumo.append({"base": base, "layout": f"ordenado_rg_{rg}", "linhas": meta.num_rows,
                       "row_groups": meta.num_row_groups, "arquivo_MB": destino.stat().st_size / 1024**2})
        del ordenado, tabela

    destino_split = experiment_dir / "parquet_variants" / base / "separado_rg_100000"
    destino_split.mkdir(parents=True, exist_ok=True)
    for referencia in ["ESC", "PROVA"]:
        parte = df.loc[df["TIPO_REFERENCIA_TERRITORIAL"].eq(referencia)]
        caminho = destino_split / f"{referencia}.parquet"
        tabela = pa.Table.from_pandas(parte, preserve_index=False)
        pq.write_table(tabela, caminho, row_group_size=100000,
                       compression="snappy", use_dictionary=True)
        meta = pq.ParquetFile(caminho).metadata
        resumo.append({"base": base, "layout": f"separado_rg_100000_{referencia}",
                       "linhas": meta.num_rows, "row_groups": meta.num_row_groups,
                       "arquivo_MB": caminho.stat().st_size / 1024**2})
        del parte, tabela
    del df

print(json.dumps({"status": "ok", "resumo": resumo}, ensure_ascii=False))
"""

In [13]:
WRITER_SCRIPT = textwrap.dedent(WRITER_SCRIPT_SOURCE)

payload_writer = {
    "project_root": str(PROJECT_ROOT),
    "experiment_dir": str(EXPERIMENT_DIR),
    "uf": UF,
    "use_threads": USE_THREADS_CONTROLE,
}
proc_writer = subprocess.run(
    [sys.executable, "-c", WRITER_SCRIPT],
    input=json.dumps(payload_writer),
    text=True,
    capture_output=True,
    cwd=PROJECT_ROOT,
    timeout=1800,
)
if proc_writer.returncode != 0:
    raise RuntimeError(f"Falha na geração das variantes.\n{proc_writer.stdout}\n{proc_writer.stderr}")
print(proc_writer.stdout)

Preparando painel ...
Preparando faixa ...
Preparando grupo ...
Preparando dsp ...
{"status": "ok", "resumo": [{"base": "painel", "layout": "ordenado_rg_50000", "linhas": 6464, "row_groups": 1, "arquivo_MB": 0.3804206848144531}, {"base": "painel", "layout": "ordenado_rg_100000", "linhas": 6464, "row_groups": 1, "arquivo_MB": 0.3804206848144531}, {"base": "painel", "layout": "ordenado_rg_250000", "linhas": 6464, "row_groups": 1, "arquivo_MB": 0.3804206848144531}, {"base": "painel", "layout": "separado_rg_100000_ESC", "linhas": 5125, "row_groups": 1, "arquivo_MB": 0.2571887969970703}, {"base": "painel", "layout": "separado_rg_100000_PROVA", "linhas": 1339, "row_groups": 1, "arquivo_MB": 0.14335346221923828}, {"base": "faixa", "layout": "ordenado_rg_50000", "linhas": 63836, "row_groups": 2, "arquivo_MB": 0.1489725112915039}, {"base": "faixa", "layout": "ordenado_rg_100000", "linhas": 63836, "row_groups": 1, "arquivo_MB": 0.14497947692871094}, {"base": "faixa", "layout": "ordenado_rg_25000

### Verificação física das variantes

In [14]:
linhas_meta = []
for base in ARQUIVOS_ORIGINAIS:
    pasta = EXPERIMENT_DIR / "parquet_variants" / base
    for caminho in sorted(pasta.glob("*.parquet")):
        pf = pq.ParquetFile(caminho)
        linhas_meta.append({"base": base, "variante": caminho.name,
                            "linhas": pf.metadata.num_rows,
                            "row_groups": pf.metadata.num_row_groups,
                            "tamanho_MB": caminho.stat().st_size / 1024**2})
    pasta_split = pasta / "separado_rg_100000"
    for caminho in sorted(pasta_split.glob("*.parquet")):
        pf = pq.ParquetFile(caminho)
        linhas_meta.append({"base": base, "variante": f"separado/{caminho.name}",
                            "linhas": pf.metadata.num_rows,
                            "row_groups": pf.metadata.num_row_groups,
                            "tamanho_MB": caminho.stat().st_size / 1024**2})

df_meta_variantes = pd.DataFrame(linhas_meta)
display(df_meta_variantes)

,base,variante,linhas,row_groups,tamanho_MB
0,painel,ordenado_rg_100000.parquet,6464,1,0.380421
1,painel,ordenado_rg_250000.parquet,6464,1,0.380421
2,painel,ordenado_rg_50000.parquet,6464,1,0.380421
3,painel,separado/ESC.parquet,5125,1,0.257189
4,painel,separado/PROVA.parquet,1339,1,0.143353
5,faixa,ordenado_rg_100000.parquet,63836,1,0.144979
6,faixa,ordenado_rg_250000.parquet,63836,1,0.144979
7,faixa,ordenado_rg_50000.parquet,63836,2,0.148973
8,faixa,separado/ESC.parquet,40659,1,0.093729
9,faixa,separado/PROVA.parquet,23177,1,0.057025


### Comparação de leitura: original × ordenado × arquivos separados

O worker aplica o filtro ao arquivo original e às variantes ordenadas. Na variante separada, lê diretamente o arquivo ESC ou PROVA, sem filtro de referência. Todas as variantes passam pela mesma harmonização municipal e geram assinaturas para validar equivalência.

In [15]:
LAYOUTS_TESTE = [
    "original",
    "ordenado_rg_50000",
    "ordenado_rg_100000",
    "ordenado_rg_250000",
    "separado_rg_100000",
]
resultados_layout = []
for ref in REFERENCIAS:
    for layout in LAYOUTS_TESTE:
        for repeticao in range(1, REPETICOES + 1):
            print(f"{ref} | {layout} | repetição {repeticao}/{REPETICOES}")
            r = executar_worker(
                referencia=ref,
                use_threads=USE_THREADS_CONTROLE,
                layout=layout,
            )
            r["cenario"] = layout
            r["repeticao"] = repeticao
            resultados_layout.append(r)

layout_detalhe, layout_resumo = resumir_resultados(resultados_layout)
display(layout_detalhe)
display(layout_resumo)

ESC | original | repetição 1/2
ESC | original | repetição 2/2
ESC | ordenado_rg_50000 | repetição 1/2
ESC | ordenado_rg_50000 | repetição 2/2
ESC | ordenado_rg_100000 | repetição 1/2
ESC | ordenado_rg_100000 | repetição 2/2
ESC | ordenado_rg_250000 | repetição 1/2
ESC | ordenado_rg_250000 | repetição 2/2
ESC | separado_rg_100000 | repetição 1/2
ESC | separado_rg_100000 | repetição 2/2
PROVA | original | repetição 1/2
PROVA | original | repetição 2/2
PROVA | ordenado_rg_50000 | repetição 1/2
PROVA | ordenado_rg_50000 | repetição 2/2
PROVA | ordenado_rg_100000 | repetição 1/2
PROVA | ordenado_rg_100000 | repetição 2/2
PROVA | ordenado_rg_250000 | repetição 1/2
PROVA | ordenado_rg_250000 | repetição 2/2
PROVA | separado_rg_100000 | repetição 1/2
PROVA | separado_rg_100000 | repetição 2/2


,cenario,referencia,repeticao,USS_apos_carga_MB,RSS_apos_carga_MB,USS_apos_gc_MB,Arrow_pool_MB,tempo_s,deep_total_MB,backend
0,original,ESC,1,372.371094,471.171875,365.417969,19.972229,0.431747,58.853294,mimalloc
1,original,ESC,2,374.910156,473.542969,367.949219,19.972229,0.419195,58.853294,mimalloc
2,ordenado_rg_50000,ESC,1,374.656250,472.863281,367.500000,19.972229,0.347207,58.853294,mimalloc
3,ordenado_rg_50000,ESC,2,369.898438,467.902344,362.917969,19.972229,0.420766,58.853294,mimalloc
4,ordenado_rg_100000,ESC,1,355.546875,453.671875,348.562500,19.972229,0.457860,58.853294,mimalloc
5,ordenado_rg_100000,ESC,2,361.804688,460.136719,354.832031,19.972229,0.442060,58.853294,mimalloc
6,ordenado_rg_250000,ESC,1,349.500000,448.113281,342.531250,19.972229,0.431721,58.853294,mimalloc
7,ordenado_rg_250000,ESC,2,367.429688,465.503906,360.238281,19.972229,0.409544,58.853294,mimalloc
8,separado_rg_100000,ESC,1,351.707031,449.535156,344.527344,19.972229,0.379804,58.853294,mimalloc
9,separado_rg_100000,ESC,2,337.738281,435.722656,330.562500,19.972229,0.370702,58.853294,mimalloc


,cenario,referencia,USS_mediana_MB,USS_min_MB,USS_max_MB,RSS_mediana_MB,USS_apos_gc_mediana_MB,Arrow_pool_mediana_MB,tempo_mediano_s,deep_total_mediana_MB,n
0,ordenado_rg_100000,ESC,358.675781,355.546875,361.804688,456.904297,351.697266,19.972229,0.449960,58.853294,2
1,ordenado_rg_100000,PROVA,272.984375,271.832031,274.136719,370.980469,275.021484,20.808838,0.274887,42.586659,2
2,ordenado_rg_250000,ESC,358.464844,349.500000,367.429688,456.808594,351.384766,19.972229,0.420633,58.853294,2
3,ordenado_rg_250000,PROVA,280.136719,278.093750,282.179688,378.113281,280.187500,20.807922,0.315549,42.586659,2
4,ordenado_rg_50000,ESC,372.277344,369.898438,374.656250,470.382812,365.208984,19.972229,0.383987,58.853294,2
5,ordenado_rg_50000,PROVA,302.246094,296.300781,308.191406,400.283203,304.281250,20.809570,0.278085,42.586659,2
6,original,ESC,373.640625,372.371094,374.910156,472.357422,366.683594,19.972229,0.425471,58.853294,2
7,original,PROVA,326.890625,324.917969,328.863281,425.103516,325.900391,20.809082,0.308280,42.586669,2
8,separado_rg_100000,ESC,344.722656,337.738281,351.707031,442.628906,337.544922,19.972229,0.375253,58.853294,2
9,separado_rg_100000,PROVA,242.958984,237.941406,247.976562,340.595703,246.984375,20.696228,0.260741,42.387752,2


### Validação de equivalência das variantes

A assinatura usa quantidade de linhas, soma e XOR dos hashes por linha. Como a ordenação física muda a ordem dos registros, soma e XOR permitem comparar o conteúdo sem exigir a mesma sequência de linhas. É uma verificação de consistência prática; não substitui uma auditoria coluna a coluna em caso de divergência.

In [16]:
validacoes = []
for ref in REFERENCIAS:
    orig = next(r for r in resultados_layout if r["referencia"] == ref and r["layout"] == "original")
    for layout in LAYOUTS_TESTE:
        alt = next(r for r in resultados_layout if r["referencia"] == ref and r["layout"] == layout)
        iguais = True
        divergencias = []
        for base in ["painel", "faixa", "grupo", "dsp"]:
            a = orig["assinaturas"][base]
            b = alt["assinaturas"][base]
            if a != b:
                iguais = False
                divergencias.append(base)
        validacoes.append({"referencia": ref, "layout": layout,
                           "equivalente": iguais,
                           "bases_divergentes": ", ".join(divergencias)})

df_validacao_layout = pd.DataFrame(validacoes)
display(df_validacao_layout)

,referencia,layout,equivalente,bases_divergentes
0,ESC,original,True,
1,ESC,ordenado_rg_50000,True,
2,ESC,ordenado_rg_100000,True,
3,ESC,ordenado_rg_250000,True,
4,ESC,separado_rg_100000,True,
5,PROVA,original,True,
6,PROVA,ordenado_rg_50000,True,
7,PROVA,ordenado_rg_100000,True,
8,PROVA,ordenado_rg_250000,True,
9,PROVA,separado_rg_100000,True,


## 7. Experimento 6 — consumo real com Gunicorn e Dash

Esta seção é separada das medições anteriores. Ela inicia a aplicação localmente com **um único worker**, consulta a página inicial e registra RSS/USS do master e do worker.

- O servidor só será iniciado quando `EXECUTAR_GUNICORN = True`.
- O comando pressupõe que `app.py` expõe `server`, como no comando já utilizado no projeto (`gunicorn app:server`).
- O teste não navega por todas as páginas nem dispara todos os callbacks. Para medir o uso real em cada visualização, será necessário navegar manualmente no dashboard enquanto a coleta estiver ativa.

In [17]:
import urllib.request
import urllib.error
import signal

GUNICORN_HOST = "127.0.0.1"
GUNICORN_PORT = 8050
GUNICORN_WORKERS = 1
EXECUTAR_GUNICORN = False
GUNICORN_LOG = EXPERIMENT_DIR / "gunicorn_diagnostico07.log"

print("Gunicorn habilitado:", EXECUTAR_GUNICORN)
print("Comando previsto:",
      f"{sys.executable} -m gunicorn app:server --bind {GUNICORN_HOST}:{GUNICORN_PORT} --workers {GUNICORN_WORKERS}")

Gunicorn habilitado: False
Comando previsto: /home/akel/anaconda2025/envs/data_science/bin/python -m gunicorn app:server --bind 127.0.0.1:8050 --workers 1


In [18]:
def memoria_arvore_processo(pid):
    try:
        raiz = psutil.Process(pid)
        processos = [raiz] + raiz.children(recursive=True)
    except psutil.Error:
        return []
    saida = []
    for proc in processos:
        try:
            m = proc.memory_full_info()
            saida.append({
                "pid": proc.pid,
                "nome": proc.name(),
                "rss_mb": m.rss / 1024**2,
                "uss_mb": getattr(m, "uss", float("nan")) / 1024**2,
                "pss_mb": getattr(m, "pss", float("nan")) / 1024**2,
                "status": proc.status(),
            })
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    return saida


def executar_teste_gunicorn(tempo_espera=30):
    if not EXECUTAR_GUNICORN:
        print("Execução desabilitada. Altere EXECUTAR_GUNICORN = True para iniciar o servidor.")
        return None

    comando = [
        sys.executable, "-m", "gunicorn", "app:server",
        "--bind", f"{GUNICORN_HOST}:{GUNICORN_PORT}",
        "--workers", str(GUNICORN_WORKERS),
        "--timeout", "120",
        "--access-logfile", "-",
        "--error-logfile", "-",
    ]
    with open(GUNICORN_LOG, "w", encoding="utf-8") as log:
        proc = subprocess.Popen(
            comando, cwd=PROJECT_ROOT, stdout=log, stderr=subprocess.STDOUT,
            env={**os.environ, "PYTHONUNBUFFERED": "1"},
        )

    processo = psutil.Process(proc.pid)
    pronto = False
    inicio = time.time()
    while time.time() - inicio < tempo_espera:
        if proc.poll() is not None:
            raise RuntimeError(f"Gunicorn encerrou com código {proc.returncode}. Consulte {GUNICORN_LOG}")
        try:
            with urllib.request.urlopen(f"http://{GUNICORN_HOST}:{GUNICORN_PORT}/", timeout=2) as resp:
                pronto = 200 <= resp.status < 500
                if pronto:
                    break
        except Exception:
            time.sleep(1)

    if not pronto:
        proc.terminate()
        raise TimeoutError(f"Gunicorn não respondeu em {tempo_espera}s. Consulte {GUNICORN_LOG}")

    time.sleep(3)
    medidas = memoria_arvore_processo(proc.pid)
    totais = {
        "processos": len(medidas),
        "RSS_soma_MB": sum(x["rss_mb"] for x in medidas),
        "USS_soma_MB": sum(x["uss_mb"] for x in medidas),
        "PSS_soma_MB": sum(x["pss_mb"] for x in medidas if np.isfinite(x["pss_mb"])),
    }
    return {"popen": proc, "processos": medidas, "totais": totais, "log": str(GUNICORN_LOG)}


teste_gunicorn = executar_teste_gunicorn()
if teste_gunicorn:
    display(pd.DataFrame(teste_gunicorn["processos"]))
    display(pd.DataFrame([teste_gunicorn["totais"]]))
    print("Log:", teste_gunicorn["log"])

Execução desabilitada. Altere EXECUTAR_GUNICORN = True para iniciar o servidor.


### Navegação e encerramento do servidor

Com o servidor ativo, abra `http://127.0.0.1:8050` no navegador, navegue pelas páginas e acione as visualizações que carregam as bases. Execute a célula abaixo durante ou depois da navegação para registrar novamente a memória da árvore de processos.

In [19]:
if "teste_gunicorn" in globals() and teste_gunicorn:
    proc_gunicorn = teste_gunicorn["popen"]
    if proc_gunicorn.poll() is None:
        processos_atual = memoria_arvore_processo(proc_gunicorn.pid)
        display(pd.DataFrame(processos_atual))
        print("RSS total MB:", round(sum(x["rss_mb"] for x in processos_atual), 2))
        print("USS total MB:", round(sum(x["uss_mb"] for x in processos_atual), 2))
    else:
        print("Gunicorn já foi encerrado.")
else:
    print("Nenhum teste Gunicorn ativo nesta sessão.")

Nenhum teste Gunicorn ativo nesta sessão.


In [20]:
# Encerra o servidor iniciado por este notebook.
if "teste_gunicorn" in globals() and teste_gunicorn:
    proc_gunicorn = teste_gunicorn["popen"]
    if proc_gunicorn.poll() is None:
        proc_gunicorn.terminate()
        try:
            proc_gunicorn.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc_gunicorn.kill()
            proc_gunicorn.wait(timeout=5)
        print("Gunicorn encerrado.")
    else:
        print("Gunicorn já estava encerrado.")

## 8. Consolidação e exportação dos resultados

A célula final exporta os resultados das etapas executadas. Ela não pressupõe que todos os experimentos tenham sido rodados; grava apenas os conjuntos disponíveis na sessão.

In [21]:
ARQUIVOS_EXPORTADOS = []

def exportar_df(nome, df):
    caminho = EXPERIMENT_DIR / f"{nome}.csv"
    df.to_csv(caminho, index=False)
    ARQUIVOS_EXPORTADOS.append(str(caminho))
    print(caminho)

if "threads_detalhe" in globals():
    exportar_df("threads_detalhe", threads_detalhe)
    exportar_df("threads_resumo", threads_resumo)
if "pools_resumo" in globals():
    exportar_df("memory_pool_resumo", pools_resumo)
if "dictionary_detalhe" in globals():
    exportar_df("read_dictionary_detalhe", dictionary_detalhe)
    exportar_df("read_dictionary_resumo", dictionary_resumo)
if "df_downcast" in globals():
    exportar_df("downcast_resumo", df_downcast)
if "layout_detalhe" in globals():
    exportar_df("organizacao_parquet_detalhe", layout_detalhe)
    exportar_df("organizacao_parquet_resumo", layout_resumo)
if "df_validacao_layout" in globals():
    exportar_df("validacao_organizacao_parquet", df_validacao_layout)

print("\nArquivos exportados:")
for caminho in ARQUIVOS_EXPORTADOS:
    print("-", caminho)

/home/akel/PycharmProjects/Projeto_ENEM/app_render/_diagnosticos_memoria_07/threads_detalhe.csv
/home/akel/PycharmProjects/Projeto_ENEM/app_render/_diagnosticos_memoria_07/threads_resumo.csv
/home/akel/PycharmProjects/Projeto_ENEM/app_render/_diagnosticos_memoria_07/memory_pool_resumo.csv
/home/akel/PycharmProjects/Projeto_ENEM/app_render/_diagnosticos_memoria_07/read_dictionary_detalhe.csv
/home/akel/PycharmProjects/Projeto_ENEM/app_render/_diagnosticos_memoria_07/read_dictionary_resumo.csv
/home/akel/PycharmProjects/Projeto_ENEM/app_render/_diagnosticos_memoria_07/downcast_resumo.csv
/home/akel/PycharmProjects/Projeto_ENEM/app_render/_diagnosticos_memoria_07/organizacao_parquet_detalhe.csv
/home/akel/PycharmProjects/Projeto_ENEM/app_render/_diagnosticos_memoria_07/organizacao_parquet_resumo.csv
/home/akel/PycharmProjects/Projeto_ENEM/app_render/_diagnosticos_memoria_07/validacao_organizacao_parquet.csv

Arquivos exportados:
- /home/akel/PycharmProjects/Projeto_ENEM/app_render/_diagno

## 9. Critérios para decisão

- **Threads:** aceitar `use_threads=False` somente se a redução de USS for repetível e o tempo adicional for aceitável.
- **Memory Pool:** comparar apenas backends efetivamente ativados; observar USS após carga e após `del`/`gc`, além do tempo.
- **Dictionary:** aceitar somente se reduzir o USS ou o pico de forma repetível, sem incompatibilidades nas operações do dashboard.
- **Downcasting:** avaliar economia de `deep`, erro numérico e compatibilidade dos indicadores. Não misturar essa avaliação com o pico de leitura.
- **Parquet:** exigir equivalência das assinaturas e ganho consistente nas duas referências. Não substituir arquivos de produção com base em uma única execução.
- **Gunicorn:** manter um worker durante o deploy com limite de memória baixo; considerar a soma de USS dos processos e observar separadamente o efeito da navegação/callbacks.

### Referências técnicas

- [PyArrow — `parquet.read_table`](https://arrow.apache.org/docs/python/generated/pyarrow.parquet.read_table.html): `use_threads` e `read_dictionary`.
- [PyArrow — Parquet reads](https://arrow.apache.org/docs/python/parquet.html): leitura multithread e organização dos arquivos.
- [Pandas — `read_parquet`](https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html): passagem de argumentos para o engine PyArrow.
- [Apache Arrow — Memory Pool](https://arrow.apache.org/docs/python/memory.html): backends e métricas de alocação.